# 🧬 Notebook 02 — Embeddings & FAISS Vector Store
Each comment → embedding vector → stored in FAISS for semantic retrieval.
**Key innovation:** one FAISS index per video prevents comment cross-contamination.

In [9]:
# ── Paths ──
from pathlib import Path
import pandas as pd
import numpy as np
from pprint import pprint

PROJECT_ROOT = Path(r"C:\Users\user\Documents\LLM - AI\Project\agentic_video_analysis")
DATA_DIR   = PROJECT_ROOT / "data"
PROC_DIR   = DATA_DIR / "processed"
RESULTS_DIR= DATA_DIR / "results"
INDEX_DIR  = PROJECT_ROOT / "index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Paths ready.")

Paths ready.


In [10]:
# ── Load cleaned dataset ──
pq = PROC_DIR / "comments_dataset_clean.parquet"
cv = PROC_DIR / "comments_dataset_clean.csv"
if pq.exists():
    df = pd.read_parquet(pq); print("Loaded parquet:", df.shape)
elif cv.exists():
    df = pd.read_csv(cv);     print("Loaded CSV:",     df.shape)
else:
    raise FileNotFoundError("Run Notebook 01 first.")
print("Columns:", df.columns.tolist())

Loaded parquet: (35409, 12)
Columns: ['text', 'video_id', 'query', 'title', 'channel', 'category', 'views', 'likes', 'published', 'category_name', 'comment_count', 'published_at']


In [11]:
# ── Filter: keep videos with >= 10 comments ──
MIN_COMMENTS = 10
vc = df.groupby("video_id").size()
valid_ids = vc[vc >= MIN_COMMENTS].index
df_valid = df[df["video_id"].isin(valid_ids)].copy()
print(f"Videos kept: {df_valid['video_id'].nunique()} | Rows: {len(df_valid)}")

Videos kept: 285 | Rows: 35344


In [12]:
# ── Load embedding model ──
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL, model_kwargs={"device":"cpu"})
print("Embedding model ready:", EMB_MODEL)

C:\Users\user\anaconda3\envs\llm_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model ready: sentence-transformers/all-MiniLM-L6-v2


In [13]:
# ── Helper: build FAISS store for ONE video ──
def build_video_store(df_src, video_id, emb):
    sub = df_src[df_src["video_id"] == video_id].copy()
    sub = sub.dropna(subset=["text"])
    sub = sub[sub["text"].str.len() > 10]
    sub = sub.drop_duplicates(subset=["text"])
    if sub.empty:
        raise ValueError(f"No usable comments for {video_id}")
    docs = []
    for _, row in sub.iterrows():
        docs.append(Document(
            page_content=str(row["text"]),
            metadata={
                "video_id":     str(row["video_id"]),
                "title":        str(row.get("title",""))    if pd.notna(row.get("title"))    else "",
                "channel":      str(row.get("channel",""))  if pd.notna(row.get("channel"))  else "",
                "category_name":str(row.get("category_name","")) if pd.notna(row.get("category_name")) else "",
                "likes":        int(row.get("likes",0))     if pd.notna(row.get("likes"))    else 0,
                "views":        int(row.get("views",0))     if pd.notna(row.get("views"))    else 0,
            }
        ))
    return FAISS.from_documents(docs, emb)

In [14]:
# ── Build + save a FAISS index for every video ──
VIDEO_IDX_DIR = INDEX_DIR / "per_video"
VIDEO_IDX_DIR.mkdir(parents=True, exist_ok=True)

all_ids = df_valid.groupby("video_id").size().sort_values(ascending=False).index.tolist()
print(f"Building indexes for {len(all_ids)} videos…\n")

errors = []
for vid_id in all_ids:
    path = VIDEO_IDX_DIR / vid_id
    if path.exists():
        print(f"  ⏩ Skip {vid_id} — already indexed")
        continue
    try:
        vs = build_video_store(df_valid, vid_id, embeddings)
        vs.save_local(str(path))
        n = len(df_valid[df_valid["video_id"]==vid_id])
        print(f"  ✅ {vid_id} — {n} comments")
    except Exception as e:
        errors.append((vid_id, str(e)))
        print(f"  ❌ {vid_id} — {e}")

print(f"\nDone. {len(all_ids)-len(errors)} OK  |  {len(errors)} errors")

Building indexes for 285 videos…

  ⏩ Skip 0Tch0N5nsRU — already indexed
  ⏩ Skip 9VlvbpXwLJs — already indexed
  ⏩ Skip YZzPCXME-QY — already indexed
  ⏩ Skip VhiXEx5idVY — already indexed
  ⏩ Skip 1sfmZTrijDQ — already indexed
  ⏩ Skip WETxkRr6dV4 — already indexed
  ⏩ Skip YJQSuUZdcV4 — already indexed
  ⏩ Skip 2EGn3rJB7DI — already indexed
  ⏩ Skip JL_grPUnXzY — already indexed
  ⏩ Skip 4lAlWPuuf2c — already indexed
  ⏩ Skip lWLnm1RCg_g — already indexed
  ⏩ Skip gWHxRuD57ig — already indexed
  ⏩ Skip BL9lacx0GGw — already indexed
  ⏩ Skip gPKkIkEnZw8 — already indexed
  ⏩ Skip TtW81KUkzjg — already indexed
  ⏩ Skip KFmqEmvRfYM — already indexed
  ⏩ Skip iCYRz2XVh0Y — already indexed
  ⏩ Skip b12JrM-6DBY — already indexed
  ⏩ Skip _bF-3Dncmeo — already indexed
  ⏩ Skip 3XiMJUpliI4 — already indexed
  ⏩ Skip KanIE9Uc8Lk — already indexed
  ⏩ Skip I4nEXhXoI70 — already indexed
  ⏩ Skip uUzwssHX-Lc — already indexed
  ⏩ Skip Kg4AC-SyChw — already indexed
  ⏩ Skip PiepQKarvTY — already

In [15]:
# ── Quick retrieval test ──
def load_video_store(video_id):
    p = VIDEO_IDX_DIR / video_id
    if not p.exists():
        raise FileNotFoundError(f"No index for {video_id}")
    return FAISS.load_local(str(p), embeddings, allow_dangerous_deserialization=True)

# Test with first available video
test_vid = df_valid.groupby("video_id").size().idxmax()
test_store = load_video_store(test_vid)
results = test_store.similarity_search("informative useful explains well", k=3)
print(f"Test video: {test_vid}")
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r.page_content[:120]}")
print("\n✅ Notebook 02 complete.")

Test video: 0Tch0N5nsRU
  [1] excellent well done very helpful
  [2] on of the best videos ive seen explaining ai
  [3] great summary and super helpful appreciate you sharing this

✅ Notebook 02 complete.
